In [90]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pickle
import os
import json
import scipy.interpolate as interp
import pathlib

In [91]:

dir = r"/Volumes/ESSD/BatteryLife" if os.name == 'posix' else r"D:\BatteryLife"
content = os.listdir(dir)
print(f"###### \n ---- Raw Content ---- \n{content} \n######")
folders = [f for f in content if os.path.isdir(os.path.join(dir, f)) and (f != "Life labels" and f != "READMEs")]
print(f"###### \n ---- Battery Folders ---- \n{folders} \n######")

###### 
 ---- Raw Content ---- 
['CALB', 'CALCE', 'HNEI', 'HUST', 'ISU_ILCC', 'Life labels', 'MATR', 'MICH', 'MICH_EXP', 'NA-ion', 'README.md', 'READMEs', 'RWTH', 'SDU', 'SNL', 'Stanford', 'Stanford_2', 'Tongji', 'UL_PUR', 'XJTU', 'ZN-coin', '.DS_Store', '._.DS_Store', '._README.md'] 
######
###### 
 ---- Battery Folders ---- 
['CALB', 'CALCE', 'HNEI', 'HUST', 'ISU_ILCC', 'MATR', 'MICH', 'MICH_EXP', 'NA-ion', 'RWTH', 'SDU', 'SNL', 'Stanford', 'Stanford_2', 'Tongji', 'UL_PUR', 'XJTU', 'ZN-coin'] 
######


In [92]:

def get_length(dir):
    content = os.listdir(dir)
    length = 0
    for f in content:
        if f.startswith("._") or f.startswith(".DS_Store"):
            continue
        with open(os.path.join(dir, f), 'r') as file:
            data = json.load(file)
            length += len(data)
    return length


        

In [93]:
content = os.listdir(dir)
print(f"###### \n ---- Content ---- \n{content} \n######")
folders = [f for f in content if os.path.isdir(os.path.join(dir, f)) and (f != "Life labels" and f != "READMEs") and not f.startswith("._") and not f.startswith(".DS_Store")]
pkl_files = [[] for _ in folders]
for i, folder in enumerate(folders) :
    folder_path = os.path.join(dir, folder)
    for file in os.listdir(folder_path):
        if file.endswith('.pkl'):
            pkl_files[i].append(file)
print(len(pkl_files))

###### 
 ---- Content ---- 
['CALB', 'CALCE', 'HNEI', 'HUST', 'ISU_ILCC', 'Life labels', 'MATR', 'MICH', 'MICH_EXP', 'NA-ion', 'README.md', 'READMEs', 'RWTH', 'SDU', 'SNL', 'Stanford', 'Stanford_2', 'Tongji', 'UL_PUR', 'XJTU', 'ZN-coin', '.DS_Store', '._.DS_Store', '._README.md'] 
######
18


In [94]:
labels = os.path.join(dir, "Life labels")
def get_dict(labels):
    dict_labels = {}
    content = os.listdir(labels)
    length = 0
  
    for f in content:
        if f.startswith("._") or f.startswith(".DS_Store"):
          continue
        with open(os.path.join(labels, f), 'r') as file:
            data = json.load(file)
            if 'Tongji' in f:
               temp_dict = {}
               for k, v in data.items():
                    k_new = k.replace("#", "-")
                    temp_dict.update({k_new:v})
               data = temp_dict   
            dict_labels.update(data)
    k= list(dict_labels.keys())
    v= list(dict_labels.values())
    return k,v


In [95]:
k,v = get_dict(labels)
print(len(v))

1208


In [96]:
import numpy as np
import scipy.interpolate as interp

def get_cycle_data(data, cycle_format):
    
    arr_total = np.zeros((cycle_format, 3, 300))
    
    for i in range(cycle_format):

        raw_time = np.array(data['cycle_data'][i]['time_in_s'])
        scaled_time = raw_time - raw_time[0]
        raw_current = np.array(data['cycle_data'][i]['current_in_A'])
        raw_voltage = np.array(data['cycle_data'][i]['voltage_in_V'])
        dt = np.diff(scaled_time)
        avg_current = 0.5 * (raw_current[1:] + raw_current[:-1])
        
        cumulative_charge = np.concatenate((
            [0],
            np.cumsum(avg_current * dt)
        ))
        
        capacity_Ah = cumulative_charge / 3600.0

        t_new = np.linspace(0, scaled_time[-1], num=300)
        f_current = interp.interp1d(scaled_time, raw_current, kind='linear')
        f_voltage = interp.interp1d(scaled_time, raw_voltage, kind='linear')
        f_capacity = interp.interp1d(scaled_time, capacity_Ah, kind='linear')
        current_interpolated = f_current(t_new) / data['nominal_capacity_in_Ah']
        voltage_interpolated = f_voltage(t_new) / data['max_voltage_limit_in_V']
        capacity_interpolated = f_capacity(t_new) / data['nominal_capacity_in_Ah']
        
        arr_total[i] = np.stack(
            (current_interpolated,
             voltage_interpolated,
             capacity_interpolated),
            axis=0
        )
    
    return arr_total


In [97]:
def get_location(file, root_dir):
    for loc in pathlib.Path(root_dir).rglob(file):
        return loc

In [98]:
class BatteryLifeDataset(Dataset):
    def __init__(self, root_dir, cycle_format, transform=None, target_transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.target_transform = target_transform
        self.cycles_file, self.soh = get_dict(os.path.join(root_dir, "Life labels"))
        self.cycle_format = cycle_format
        
    def __len__(self):
        return len(self.soh)
    
    def __getitem__(self, idx):
     file_name = self.cycles_file[idx]
     path = get_location(file_name, self.root_dir)

     if path is None:
        raise ValueError(f"Missing file: {file_name}")

     with open(path, 'rb') as file:
        data = pickle.load(file)

     features = get_cycle_data(data,self.cycle_format)
     label = self.soh[idx]

     return (
        torch.tensor(features, dtype=torch.float32),
        torch.tensor(label, dtype=torch.float32)
     )


In [99]:
## Split code taken from https://www.geeksforgeeks.org/deep-learning/how-to-split-a-dataset-using-pytorch/

dataset = BatteryLifeDataset(root_dir=dir,cycle_format=1)
print(f"Size of dataset: {len(dataset)}")
total = len(dataset)
train_data, val_data = torch.utils.data.random_split(dataset, [1000, 208])
train_loader = DataLoader(train_data, batch_size=5, shuffle=False, num_workers=0)
validation_loader = DataLoader(val_data, batch_size=5, shuffle=False, num_workers=0)

Size of dataset: 1208


In [100]:

train_features, train_labels = next(iter(train_loader))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels}")
print(train_features.squeeze(1).shape)

Feature batch shape: torch.Size([5, 1, 3, 300])
Labels batch shape: tensor([ 148., 2353.,  694.,  716.,  228.])
torch.Size([5, 3, 300])


In [101]:
class RNNnetwork(nn.Module):
    def __init__(self, hidden_size=20):
        super(RNNnetwork, self).__init__()

        self.rnn = nn.RNN(
            input_size=3, hidden_size=hidden_size, num_layers=1, batch_first=True
        )

        self.out = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = x.squeeze(1).permute(0, 2, 1)
        
        _, hidden_layers = self.rnn(x)
        cycle_to_80_pred = self.out(hidden_layers[-1]) 

        return cycle_to_80_pred.flatten()


In [102]:
model = RNNnetwork(hidden_size=20)

criterion = nn.MSELoss()  
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [103]:

epochs = 50

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch_x, batch_y in train_loader:
        # batch_x: (batch, 1, 3, 300)
        # batch_y: (batch,)   -> target cycles

        optimizer.zero_grad()

        preds = model(batch_x)   # (batch,)
        loss = criterion(preds, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch} | Loss: {total_loss:.4f}")


Epoch 0 | Loss: 212722511.9219
Epoch 1 | Loss: 211128377.7969


KeyboardInterrupt: 